# Normalize annotated single cells using negative control

## Import libraries

In [3]:
import os
import pathlib
import pprint

import pandas as pd
from pycytominer import normalize
from pycytominer.cyto_utils import output
from timelapse_utils.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from timelapse_utils.profiling_utils.sc_extraction_utils import add_single_cell_count_df

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.notebook as tqdm
else:
    import tqdm

## Set paths and variables

In [4]:
# load in platemap file as a pandas dataframe
platemap_path = pathlib.Path(
    f"{root_dir}/Wave2_data/0.download_data/platemap/platemap.csv"
).resolve()

image_base_dir = bandicoot_check(
    bandicoot_mount_path=pathlib.Path(f"{os.path.expanduser('~')}/mnt/bandicoot/"),
    root_dir=root_dir,
)
image_base_dir = pathlib.Path(
    f"{image_base_dir}/live_cell_timelapse_pyroptosis_project_data/processed_data/"
).resolve(strict=True)
annotated_profiles_path = pathlib.Path(
    f"{image_base_dir}/6.annotated_profiles/annotated_profiles_wave2.parquet"
).resolve()

normalized_profiles_path = pathlib.Path(
    f"{image_base_dir}/7.normalized_profiles/normalized_profiles_wave2.parquet"
).resolve()
normalized_profiles_path.parent.mkdir(exist_ok=True)

## Normalize with standardize method with negative control on annotated data

The normalization needs to occur per time step. 
This code cell will split the data into time steps and normalize each time step separately.
Then each normalized time step will be concatenated back together. 

This last cell does not get run due to memory constraints. 
It is run on an HPC cluster with more memory available.

In [ ]:
# set the metadata conditions to fit and apply normalization to
samples = "Metadata_Inducer == 'DMSO' & Metadata_Inducer_dose == '0.15%' & Metadata_Inhibitor == 'DMSO' & Metadata_Inhibitor_dose == '0.15%' & Metadata_Time == '1'"

In [ ]:
# read in the annotated file
annotated_df = pd.read_parquet(annotated_profiles_path)
# get the features (not the metadata) to use for normalization
features = [col for col in annotated_df.columns if "metadata" not in col.lower()]
# apply normalization to the annotated df using the specified samples as the reference for normalization
normalized_df = normalize(
    # df with annotated raw merged single cell features
    profiles=annotated_df,
    features=features,
    # specify samples used as normalization reference (negative control)
    samples=samples,
    # normalization method used
    method="standardize",
)
# save the normalized profiles as a parquet file
output(
    normalized_df,
    output_filename=normalized_profiles_path,
    output_type="parquet",
)
if annotated_df.shape[0] != normalized_df.shape[0]:
    raise ValueError(
        f"Number of rows in the annotated df ({annotated_df.shape[0]}) does not match the number of rows in the normalized df ({normalized_df.shape[0]}). Please check the input annotated df and the samples used for normalization."
    )
normalized_df.head()

/home/lippincm/miniforge3/envs/timelapse_ibp_env/lib/python3.12/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/home/lippincm/miniforge3/envs/timelapse_ibp_env/lib/python3.12/site-packages/sklearn/utils/extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/home/lippincm/miniforge3/envs/timelapse_ibp_env/lib/python3.12/site-packages/sklearn/utils/extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


,Metadata_Well,Metadata_Inducer,Metadata_Inducer_dose,Metadata_Inhibitor,Metadata_Inhibitor_dose,Metadata_Time,Metadata_Well_FOV,Metadata_FOV,Metadata_Well_FOV_Time,Metadata_ImageNumber,...,Nuclei_Texture_Variance_CL640_3_02_256,Nuclei_Texture_Variance_CL640_3_03_256,Nuclei_Texture_Variance_NucleoLive_3_00_256,Nuclei_Texture_Variance_NucleoLive_3_01_256,Nuclei_Texture_Variance_NucleoLive_3_02_256,Nuclei_Texture_Variance_NucleoLive_3_03_256,Nuclei_Texture_Variance_SYTOXGreen_3_00_256,Nuclei_Texture_Variance_SYTOXGreen_3_01_256,Nuclei_Texture_Variance_SYTOXGreen_3_02_256,Nuclei_Texture_Variance_SYTOXGreen_3_03_256
0,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.618192,-0.597732,-0.118434,-0.11884,-0.118608,-0.118145,-0.053999,-0.054311,-0.05404,-0.054339
1,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.758541,-0.738964,-0.118434,-0.11884,-0.118608,-0.118145,-0.053999,-0.054311,-0.05404,-0.054339
2,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.675617,-0.643143,-0.118434,-0.11884,-0.118608,-0.118145,-0.053999,-0.054311,-0.05404,-0.054339
3,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,-0.755643,-0.735870,-0.118434,-0.11884,-0.118608,-0.118145,-0.053999,-0.054311,-0.05404,-0.054339
4,B2,DMSO,0.15%,DMSO,0.15%,1,B2_1,1,B2_1_1,1,...,0.042727,0.056584,-0.118434,-0.11884,-0.118608,-0.118145,-0.053999,-0.054311,-0.05404,-0.054339
